# AISC DeepFake — Calibrated Validation-Weighted Score-Level Fusion

Bu notebook, aynı model ailesinin **Eye + Brow + Mouth** tahminlerini üç aşamada
birleştirir:

1. **Calibration:** Bölgesel skorları validation verisi üzerinde ortak
   `P(FAKE)` ölçeğine getirir.
2. **Validation-weighting:** Her bölgenin güvenilirlik ağırlığını validation
   ROC-AUC değerinden türetir.
3. **Late fusion:** Kalibre edilmiş üç bölgesel olasılığın ağırlıklı toplamını
   final skor olarak üretir.

## Model families

- Swin V2 Tiny
- EfficientNet-B0
- Swin V2 Tiny + Texture Fusion

## Bilimsel kural

**TEST seti calibration, ağırlık veya threshold öğrenmek için kullanılmaz.**

Bir model ailesi için validation prediction bulunamazsa notebook o aileyi
`BLOCKED_MISSING_VALIDATION` olarak raporlar. Test üzerinde calibration fit ederek
sonucu yapay biçimde iyileştirmez.

Bu notebook, daha önce kullanılan V4 calibrated weighted score-fusion yaklaşımının
provenance/audit prensiplerini bizim üç ortak model ailemize uyarlar.

## Fusion formulation

Her bölge için validation skorlarından 1-boyutlu Logistic/Platt calibrator öğrenilir:

\[
\hat p_r = \sigma(a_r s_r + b_r)
\]

Validation ROC-AUC tabanlı güvenilirlik:

\[
q_r = \max(AUC_r - 0.5, \epsilon)
\]

Normalize ağırlık:

\[
w_r = \frac{q_r}{\sum_j q_j}
\]

Final skor:

\[
P_{fusion}
=
w_{eye}\hat p_{eye}
+
w_{brow}\hat p_{brow}
+
w_{mouth}\hat p_{mouth}
\]

Ana karar eşiği, yöntemler arası karşılaştırma için önceden sabit **0.50** tutulur.

In [7]:
# ============================================================
# 1) COLAB + CONFIG
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version

import json
import math
import os
import re
import sys
import warnings

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

METHOD_NAME = "03_calibrated_weighted_score_fusion"
DECISION_THRESHOLD = 0.50
FRAME_AGGREGATION = "mean"
WEIGHT_EPSILON = 1e-6

FIGURE_DPI = 600
MIN_FIGURE_SHORT_EDGE_PX = 600

OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/"
    "AISC DeepFake Çalışmaları/Deney 1/"
    "Kader/Deney 1/Sonuçlar/Fusion_Experiments"
)

SEARCH_ROOTS = {
    "eye": Path(
        "/content/drive/MyDrive/AISC DeepFake Çalışmaları/"
        "Deney 1/Kader/Deney 1/Sonuçlar"
    ),
    "brow": Path(
        "/content/drive/MyDrive/AISC DeepFake Çalışmaları/"
        "Deney 1/Nazlıcan/Deney 1/Sonuçlar"
    ),
    "mouth": Path(
        "/content/drive/MyDrive/AISC DeepFake Çalışmaları/"
        "Deney 1/Dilara/Deney 1/Sonuçlar"
    ),
}

MODEL_FAMILIES = {
    "swinv2_tiny": {
        "eye_test": (
            SEARCH_ROOTS["eye"]
            / "20260807_1031_eye_swinv2_tiny_seed42"
            / "predictions"
            / "test_frame_predictions.csv"
        ),
        "brow_test": (
            SEARCH_ROOTS["brow"]
            / "Kas_SwinV2_Tiny_Detayli_Sonuc_pdf"
            / "predictions"
            / "test_frame_predictions.csv"
        ),
        "mouth_test": (
            SEARCH_ROOTS["mouth"]
            / "20260807_2235_mouth_swinv2_tiny_seed42"
            / "predictions"
            / "test_frame_predictions.csv"
        ),
    },

    "efficientnet_b0": {
        "eye_test": (
            SEARCH_ROOTS["eye"]
            / "20260808_0803_eye_efficientnet_b0_seed42"
            / "predictions"
            / "test_predictions.csv"
        ),
        "brow_test": (
            SEARCH_ROOTS["brow"]
            / "20260808_1248_eyebrow_efficientnet_b0_seed42"
            / "predictions"
            / "test_predictions.csv"
        ),
        "mouth_test": (
            SEARCH_ROOTS["mouth"]
            / "20260808_1257_mouth_efficientnet_b0_seed42"
            / "predictions"
            / "test_predictions.csv"
        ),
    },

    "swinv2_texture": {
        "eye_test": (
            SEARCH_ROOTS["eye"]
            / "20260806_1748_eye_swinv2_texturefusion_seed42"
            / "full"
            / "predictions"
            / "test_predictions.csv"
        ),
        "brow_test": (
            SEARCH_ROOTS["brow"]
            / "Swin V2-Tiny + LBP + GLCM + Gabor + Wavelet Fusion"
            / "predictions"
            / "test_predictions_frame_level.csv"
        ),
        "mouth_test": (
            SEARCH_ROOTS["mouth"]
            / "SwinV2_TextureFusion_Mouth"
            / "20260807_1550_mouth_swinv2_texturefusion_seed42"
            / "full"
            / "predictions"
            / "test_predictions.csv"
        ),
    },
}

# ------------------------------------------------------------------
# Validation predictions
#
# SwinV2 Tiny validation predictions were already generated successfully.
# The run ID can differ if you generated them in another run, so the notebook
# discovers the latest successful generator output automatically.
#
# For any family/region, you can optionally place a verified validation CSV
# path below. Never put a TEST CSV here.
# ------------------------------------------------------------------

VALIDATION_PREDICTION_OVERRIDE = {

    "swinv2_tiny": {

        "eye": Path(
            "/content/drive/MyDrive/"
            "AISC DeepFake Çalışmaları/Deney 1/"
            "Kader/Deney 1/Sonuçlar/"
            "Fusion_Experiments/"
            "_validation_predictions_for_logistic_fusion/"
            "20260809_173503_validation_prediction_export_seed42/"
            "swinv2_tiny/eye/"
            "validation_predictions.csv"
        ),

        "brow": Path(
            "/content/drive/MyDrive/"
            "AISC DeepFake Çalışmaları/Deney 1/"
            "Kader/Deney 1/Sonuçlar/"
            "Fusion_Experiments/"
            "_validation_predictions_for_logistic_fusion/"
            "20260809_173503_validation_prediction_export_seed42/"
            "swinv2_tiny/brow/"
            "validation_predictions.csv"
        ),

        "mouth": Path(
            "/content/drive/MyDrive/"
            "AISC DeepFake Çalışmaları/Deney 1/"
            "Kader/Deney 1/Sonuçlar/"
            "Fusion_Experiments/"
            "_validation_predictions_for_logistic_fusion/"
            "20260809_173503_validation_prediction_export_seed42/"
            "swinv2_tiny/mouth/"
            "validation_predictions.csv"
        ),
    },


    "efficientnet_b0": {
        "eye": None,
        "brow": None,
        "mouth": None,
    },


    "swinv2_texture": {
        "eye": None,
        "brow": None,
        "mouth": None,
    },
}

RUN_ID = (
    datetime.now(timezone.utc)
    .strftime("%Y%m%d_%H%M%S_%f")
    + "_calibrated_weighted_fusion_seed42"
)

RUN_DIR = OUTPUT_ROOT / METHOD_NAME / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)

print("RUN ID :", RUN_ID)
print("OUTPUT :", RUN_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RUN ID : 20260809_193418_108990_calibrated_weighted_fusion_seed42
OUTPUT : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Kader/Deney 1/Sonuçlar/Fusion_Experiments/03_calibrated_weighted_score_fusion/20260809_193418_108990_calibrated_weighted_fusion_seed42


In [8]:
# ============================================================
# 2) ATOMIC OUTPUT + ENVIRONMENT
# ============================================================

import matplotlib
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)


def package_version(package_name):
    try:
        return version(package_name)
    except PackageNotFoundError:
        return "not_installed"


def json_default(value):
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(type(value).__name__)


def atomic_write_json(payload, target):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    temp = target.with_suffix(target.suffix + ".tmp")

    with temp.open("w", encoding="utf-8") as f:
        json.dump(
            payload,
            f,
            indent=2,
            ensure_ascii=False,
            default=json_default,
        )
        f.flush()
        os.fsync(f.fileno())

    with temp.open("r", encoding="utf-8") as f:
        json.load(f)

    os.replace(temp, target)


def atomic_write_csv(df, target):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    temp = target.with_suffix(target.suffix + ".tmp")

    df.to_csv(temp, index=False)
    verify = pd.read_csv(temp)

    if len(verify) != len(df):
        raise RuntimeError(
            f"CSV integrity failure: {target}"
        )

    os.replace(temp, target)


def save_figure(fig, target_stem):
    target_stem = Path(target_stem)
    target_stem.parent.mkdir(parents=True, exist_ok=True)

    png_path = target_stem.with_suffix(".png")
    svg_path = target_stem.with_suffix(".svg")

    temp_png = png_path.with_suffix(".png.tmp")
    temp_svg = svg_path.with_suffix(".svg.tmp")

    fig.savefig(
        temp_png,
        format="png",
        dpi=FIGURE_DPI,
        bbox_inches="tight",
    )

    fig.savefig(
        temp_svg,
        format="svg",
        bbox_inches="tight",
    )

    with Image.open(temp_png) as image:
        if min(image.size) < MIN_FIGURE_SHORT_EDGE_PX:
            raise RuntimeError(
                f"Figure below minimum resolution: "
                f"{image.size}"
            )

    os.replace(temp_png, png_path)
    os.replace(temp_svg, svg_path)
    plt.close(fig)


ENVIRONMENT = {
    "run_id": RUN_ID,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "method": METHOD_NAME,
    "python": sys.version,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
    "scikit_learn": package_version("scikit-learn"),
    "decision_threshold": DECISION_THRESHOLD,
    "frame_aggregation": FRAME_AGGREGATION,
    "calibration": "1D logistic / Platt scaling on validation only",
    "weight_source": "validation ROC-AUC only",
    "test_used_for_calibration": False,
    "test_used_for_weight_selection": False,
    "test_used_for_threshold_selection": False,
}

atomic_write_json(
    ENVIRONMENT,
    RUN_DIR / "environment.json",
)

In [9]:
# ============================================================
# 3) ROBUST PREDICTION SCHEMA + FRAME KEYS
# ============================================================

LABEL_CANDIDATES = [
    "true_label",
    "label",
    "label_int",
    "target",
    "y_true",
    "ground_truth",
    "class_id",
    "true_class",
]

PROB_CANDIDATES = [
    "fake_probability",
    "prob_fake",
    "probability_fake",
    "probability",
    "prob",
    "y_score",
    "score",
    "fake_prob",
    "prediction_probability",
]

KEY_CANDIDATES = [
    "source_frame",
    "relative_frame_path",
    "frame_path",
    "original_frame",
    "image_path",
    "path",
    "sample_id",
    "frame_stem",
]


def first_existing(columns, candidates):
    lower_map = {
        str(column).lower(): column
        for column in columns
    }

    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]

    return None


def normalize_label(value):
    if pd.isna(value):
        return np.nan

    if isinstance(
        value,
        (int, float, np.integer, np.floating),
    ):
        return int(float(value) >= 0.5)

    value = str(value).strip().lower()

    if value in {
        "1",
        "fake",
        "deepfake",
        "manipulated",
        "sahte",
        "f",
    }:
        return 1

    if value in {
        "0",
        "real",
        "genuine",
        "original",
        "gerçek",
        "r",
    }:
        return 0

    return int(float(value) >= 0.5)


def canonical_frame_key(value):
    """
    Region-specific names -> same upstream frame ID.

    Examples:
      fake_test_00000__face_00.jpg
      fake_test_00000.jpg
      fake_test_fake_test_00000_face00.png
    all become:
      fake_test_00000
    """

    if pd.isna(value):
        return None

    text = (
        str(value)
        .replace("\\", "/")
        .strip()
        .lower()
    )

    filename = text.split("/")[-1]

    filename = re.sub(
        r"\.(jpg|jpeg|png|bmp|webp|npy)$",
        "",
        filename,
        flags=re.IGNORECASE,
    )

    matches = re.findall(
        r"(real|fake)_(train|test|val|validation)_(\d+)",
        filename,
        flags=re.IGNORECASE,
    )

    if not matches:
        return None

    label, split, number = matches[-1]

    split = split.lower()

    if split == "validation":
        split = "val"

    number = str(int(number)).zfill(5)

    return f"{label.lower()}_{split}_{number}"

In [10]:
# ============================================================
# 4) COMPANION METADATA RESOLUTION FOR CACHE/HASH PATHS
# ============================================================

SOURCE_KEY_CANDIDATES = [
    "source_frame",
    "relative_frame_path",
    "input_path",
    "input_relative_path",
    "original_frame",
    "frame_stem",
    "image_path",
    "path",
]


def find_companion_metadata(prediction_path):
    prediction_path = Path(prediction_path)

    names = [
        "eligible_metadata.csv",
        "eligible_metadata_before_cache.csv",
        "metadata_used.csv",
        "validated_training_manifest.csv",
        "training_manifest.csv",
    ]

    current = prediction_path.parent
    candidates = []

    for _ in range(7):
        for name in names:
            candidates.append(current / "artifacts" / name)
            candidates.append(current / "audit" / name)
            candidates.append(current / name)

        if current.parent == current:
            break

        current = current.parent

    for candidate in candidates:
        if candidate.is_file():
            try:
                df = pd.read_csv(candidate, nrows=5)
            except Exception:
                continue

            if "sample_id" in df.columns:
                return candidate

    return None


def build_key_from_metadata(
    prediction_df,
    prediction_path,
    region_name,
):
    if "sample_id" not in prediction_df.columns:
        raise ValueError(
            f"{region_name}: cache/hash prediction without sample_id."
        )

    metadata_path = find_companion_metadata(
        prediction_path
    )

    if metadata_path is None:
        raise FileNotFoundError(
            f"{region_name}: companion metadata not found for "
            f"{prediction_path}"
        )

    metadata = pd.read_csv(metadata_path)

    source_column = None
    source_keys = None

    for candidate in SOURCE_KEY_CANDIDATES:
        if candidate not in metadata.columns:
            continue

        keys = metadata[candidate].map(
            canonical_frame_key
        )

        if float(keys.notna().mean()) >= 0.95:
            source_column = candidate
            source_keys = keys
            break

    if source_column is None:
        raise ValueError(
            f"{region_name}: no trustworthy source-frame field "
            f"in {metadata_path}"
        )

    mapping = pd.DataFrame({
        "sample_id": (
            metadata["sample_id"]
            .astype(str)
            .str.strip()
        ),
        "fusion_key": source_keys,
    })

    mapping = (
        mapping
        .dropna(subset=["fusion_key"])
        .drop_duplicates(subset=["sample_id"])
    )

    ids = (
        prediction_df["sample_id"]
        .astype(str)
        .str.strip()
    )

    merged = (
        pd.DataFrame({"sample_id": ids})
        .merge(
            mapping,
            on="sample_id",
            how="left",
            validate="many_to_one",
        )
    )

    missing = int(
        merged["fusion_key"].isna().sum()
    )

    if missing:
        raise ValueError(
            f"{region_name}: {missing} sample IDs could not "
            "be mapped to source frames."
        )

    return merged["fusion_key"], {
        "key_resolution": "companion_metadata",
        "metadata_path": str(metadata_path),
        "metadata_source_column": source_column,
    }


def resolve_fusion_key(
    raw,
    prediction_path,
    region_name,
):
    for column in KEY_CANDIDATES:
        if column not in raw.columns:
            continue

        keys = raw[column].map(
            canonical_frame_key
        )

        if float(keys.notna().mean()) >= 0.95:
            return keys, {
                "key_resolution": "prediction_column",
                "key_column": column,
            }

    return build_key_from_metadata(
        raw,
        prediction_path,
        region_name,
    )

In [11]:
# ============================================================
# 5) LOAD + AGGREGATE ONE REGION
# ============================================================

def load_region_predictions(
    path,
    region_name,
):
    if path is None:
        raise FileNotFoundError(
            f"{region_name}: prediction path is None."
        )

    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(
            f"{region_name}: prediction CSV missing:\n{path}"
        )

    raw = pd.read_csv(path)

    if raw.empty:
        raise ValueError(
            f"{region_name}: empty prediction CSV."
        )

    label_col = first_existing(
        raw.columns,
        LABEL_CANDIDATES,
    )

    score_col = first_existing(
        raw.columns,
        PROB_CANDIDATES,
    )

    if label_col is None:
        raise ValueError(
            f"{region_name}: label column not found. "
            f"Columns={list(raw.columns)}"
        )

    if score_col is None:
        raise ValueError(
            f"{region_name}: probability/score column not found. "
            f"Columns={list(raw.columns)}"
        )

    labels = raw[label_col].map(
        normalize_label
    )

    scores = pd.to_numeric(
        raw[score_col],
        errors="coerce",
    )

    if labels.isna().any():
        raise ValueError(
            f"{region_name}: invalid labels."
        )

    if scores.isna().any():
        raise ValueError(
            f"{region_name}: invalid scores."
        )

    # Current three model families store probabilities in [0,1].
    # The calibration layer still learns a validation mapping.
    if ((scores < 0) | (scores > 1)).any():
        raise ValueError(
            f"{region_name}: expected probability-like scores "
            "in [0,1] for these model families."
        )

    fusion_keys, key_audit = resolve_fusion_key(
        raw,
        path,
        region_name,
    )

    normalized = pd.DataFrame({
        "fusion_key": fusion_keys,
        f"label_{region_name}": labels.astype(int),
        f"score_{region_name}": scores.astype(float),
    })

    if normalized["fusion_key"].isna().any():
        raise ValueError(
            f"{region_name}: unresolved frame keys."
        )

    grouped = (
        normalized
        .groupby("fusion_key", as_index=False)
        .agg(
            **{
                f"label_min_{region_name}": (
                    f"label_{region_name}",
                    "min",
                ),
                f"label_max_{region_name}": (
                    f"label_{region_name}",
                    "max",
                ),
                f"score_{region_name}": (
                    f"score_{region_name}",
                    FRAME_AGGREGATION,
                ),
                f"roi_count_{region_name}": (
                    f"score_{region_name}",
                    "size",
                ),
            }
        )
    )

    conflict = (
        grouped[f"label_min_{region_name}"]
        != grouped[f"label_max_{region_name}"]
    )

    if conflict.any():
        raise ValueError(
            f"{region_name}: conflicting labels inside upstream frame."
        )

    grouped[f"label_{region_name}"] = (
        grouped[f"label_min_{region_name}"].astype(int)
    )

    grouped = grouped.drop(
        columns=[
            f"label_min_{region_name}",
            f"label_max_{region_name}",
        ]
    )

    audit = {
        "region": region_name,
        "path": str(path),
        "source_rows": int(len(raw)),
        "unique_frames": int(len(grouped)),
        "score_column": str(score_col),
        "label_column": str(label_col),
        "multi_roi_frames": int(
            (grouped[f"roi_count_{region_name}"] > 1).sum()
        ),
        **key_audit,
    }

    return grouped, audit


def align_three(
    eye_path,
    brow_path,
    mouth_path,
):
    eye, eye_audit = load_region_predictions(
        eye_path,
        "eye",
    )

    brow, brow_audit = load_region_predictions(
        brow_path,
        "brow",
    )

    mouth, mouth_audit = load_region_predictions(
        mouth_path,
        "mouth",
    )

    aligned = (
        eye
        .merge(
            brow,
            on="fusion_key",
            how="inner",
            validate="one_to_one",
        )
        .merge(
            mouth,
            on="fusion_key",
            how="inner",
            validate="one_to_one",
        )
    )

    if aligned.empty:
        raise RuntimeError(
            "Eye/Brow/Mouth common-frame intersection is empty."
        )

    labels = aligned[
        [
            "label_eye",
            "label_brow",
            "label_mouth",
        ]
    ].astype(int)

    consistent = labels.nunique(axis=1).eq(1)

    if not consistent.all():
        raise ValueError(
            f"{int((~consistent).sum())} aligned frames "
            "have inconsistent labels."
        )

    aligned["label"] = (
        labels.iloc[:, 0].astype(int)
    )

    aligned = (
        aligned
        .sort_values("fusion_key")
        .reset_index(drop=True)
    )

    audit = {
        "counts": {
            "eye_frames": int(len(eye)),
            "brow_frames": int(len(brow)),
            "mouth_frames": int(len(mouth)),
            "common_frames": int(len(aligned)),
            "real_common_frames": int(
                (aligned["label"] == 0).sum()
            ),
            "fake_common_frames": int(
                (aligned["label"] == 1).sum()
            ),
        },
        "eye": eye_audit,
        "brow": brow_audit,
        "mouth": mouth_audit,
    }

    return aligned, audit

In [13]:
# ============================================================
# 6) VALIDATION PREDICTION PATHS — DIRECT / VERIFIED
# ============================================================

from pathlib import Path
import pandas as pd

VALIDATION_RUN_ROOT = Path(
    "/content/drive/MyDrive/"
    "AISC DeepFake Çalışmaları/Deney 1/"
    "Kader/Deney 1/Sonuçlar/"
    "Fusion_Experiments/"
    "_validation_predictions_for_logistic_fusion/"
    "20260809_173503_validation_prediction_export_seed42"
)


VALIDATION_PATHS = {

    # ========================================================
    # 1) SWIN V2 TINY — VALIDATION FILES ALREADY GENERATED
    # ========================================================

    "swinv2_tiny": {

        "eye": (
            VALIDATION_RUN_ROOT
            / "swinv2_tiny"
            / "eye"
            / "validation_predictions.csv"
        ),

        "brow": (
            VALIDATION_RUN_ROOT
            / "swinv2_tiny"
            / "brow"
            / "validation_predictions.csv"
        ),

        "mouth": (
            VALIDATION_RUN_ROOT
            / "swinv2_tiny"
            / "mouth"
            / "validation_predictions.csv"
        ),
    },


    # ========================================================
    # 2) EFFICIENTNET-B0 — NOT GENERATED YET
    # ========================================================

    "efficientnet_b0": {
        "eye": None,
        "brow": None,
        "mouth": None,
    },


    # ========================================================
    # 3) SWIN V2 + TEXTURE — NOT GENERATED YET
    # ========================================================

    "swinv2_texture": {
        "eye": None,
        "brow": None,
        "mouth": None,
    },
}


# ============================================================
# VALIDATION PATH QUALITY GATE
# ============================================================

validation_rows = []

print("\n" + "=" * 100)
print("VALIDATION PREDICTION FILE CHECK")
print("=" * 100)


for family, region_paths in VALIDATION_PATHS.items():

    print(f"\n[{family}]")

    for region in [
        "eye",
        "brow",
        "mouth",
    ]:

        path = region_paths[region]

        if path is None:

            status = "NOT AVAILABLE"

            validation_rows.append(
                {
                    "model_family": family,
                    "region": region,
                    "validation_path": None,
                    "source": "not_available",
                    "exists": False,
                }
            )

            print(
                f"{region.upper():5s} | "
                f"⚪ {status}"
            )

            continue


        path = Path(path)

        exists = path.is_file()


        if exists:

            # ------------------------------------------------
            # CSV sanity check
            # ------------------------------------------------

            df_check = pd.read_csv(path)

            if df_check.empty:
                raise RuntimeError(
                    f"{family}/{region}: "
                    f"validation CSV exists but is empty:\n"
                    f"{path}"
                )

            print(
                f"{region.upper():5s} | "
                f"✅ FOUND | "
                f"rows={len(df_check)}"
            )

            print(
                f"      {path}"
            )


        else:

            print(
                f"{region.upper():5s} | "
                f"❌ MISSING"
            )

            print(
                f"      {path}"
            )


        validation_rows.append(
            {
                "model_family": family,
                "region": region,
                "validation_path": (
                    str(path)
                    if exists
                    else None
                ),
                "source": (
                    "generated_validation_export"
                    if exists
                    else "configured_but_missing"
                ),
                "exists": bool(exists),
            }
        )


# ============================================================
# DISCOVERY/AUDIT TABLE
# ============================================================

validation_discovery = pd.DataFrame(
    validation_rows
)


atomic_write_csv(
    validation_discovery,
    RUN_DIR
    / "audit"
    / "validation_prediction_discovery.csv",
)


print("\n" + "=" * 100)
print("SUMMARY")
print("=" * 100)


display(
    validation_discovery[
        [
            "model_family",
            "region",
            "validation_path",
            "source",
            "exists",
        ]
    ]
)


# ============================================================
# SWIN V2 TINY HARD QUALITY GATE
# ============================================================

for region in [
    "eye",
    "brow",
    "mouth",
]:

    path = VALIDATION_PATHS[
        "swinv2_tiny"
    ][region]

    if not Path(path).is_file():

        raise FileNotFoundError(
            "\n"
            "SwinV2 Tiny validation prediction is missing.\n"
            f"Region: {region}\n"
            f"Expected path:\n{path}"
        )


print("\n✅ SWINV2 TINY: ALL 3 VALIDATION PREDICTION FILES ARE READY.")


VALIDATION PREDICTION FILE CHECK

[swinv2_tiny]
EYE   | ❌ MISSING
      /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Kader/Deney 1/Sonuçlar/Fusion_Experiments/_validation_predictions_for_logistic_fusion/20260809_173503_validation_prediction_export_seed42/swinv2_tiny/eye/validation_predictions.csv
BROW  | ❌ MISSING
      /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Kader/Deney 1/Sonuçlar/Fusion_Experiments/_validation_predictions_for_logistic_fusion/20260809_173503_validation_prediction_export_seed42/swinv2_tiny/brow/validation_predictions.csv
MOUTH | ❌ MISSING
      /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Kader/Deney 1/Sonuçlar/Fusion_Experiments/_validation_predictions_for_logistic_fusion/20260809_173503_validation_prediction_export_seed42/swinv2_tiny/mouth/validation_predictions.csv

[efficientnet_b0]
EYE   | ⚪ NOT AVAILABLE
BROW  | ⚪ NOT AVAILABLE
MOUTH | ⚪ NOT AVAILABLE

[swinv2_texture]
EYE   | ⚪ NOT AVAILABLE
BROW  | ⚪ NOT AVAILABLE
MOUTH

,model_family,region,validation_path,source,exists
0,swinv2_tiny,eye,None,configured_but_missing,False
1,swinv2_tiny,brow,None,configured_but_missing,False
2,swinv2_tiny,mouth,None,configured_but_missing,False
3,efficientnet_b0,eye,None,not_available,False
4,efficientnet_b0,brow,None,not_available,False
5,efficientnet_b0,mouth,None,not_available,False
6,swinv2_texture,eye,None,not_available,False
7,swinv2_texture,brow,None,not_available,False
8,swinv2_texture,mouth,None,not_available,False


FileNotFoundError: 
SwinV2 Tiny validation prediction is missing.
Region: eye
Expected path:
/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Kader/Deney 1/Sonuçlar/Fusion_Experiments/_validation_predictions_for_logistic_fusion/20260809_173503_validation_prediction_export_seed42/swinv2_tiny/eye/validation_predictions.csv

In [ ]:
# ============================================================
# 7) CALIBRATION + VALIDATION WEIGHTS
# ============================================================

def fit_platt_calibrator(
    scores,
    labels,
):
    """
    1D logistic calibration.
    Fit ONLY on validation predictions.
    """

    scores = np.asarray(
        scores,
        dtype=float,
    ).reshape(-1, 1)

    labels = np.asarray(
        labels,
        dtype=int,
    )

    if len(np.unique(labels)) != 2:
        raise ValueError(
            "Calibration requires both REAL and FAKE."
        )

    model = LogisticRegression(
        C=1.0,
        solver="lbfgs",
        max_iter=2000,
        random_state=SEED,
    )

    model.fit(
        scores,
        labels,
    )

    return model


def calibrate(
    model,
    scores,
):
    scores = np.asarray(
        scores,
        dtype=float,
    ).reshape(-1, 1)

    probability = (
        model
        .predict_proba(scores)[:, 1]
    )

    if not np.isfinite(
        probability
    ).all():
        raise RuntimeError(
            "Non-finite calibrated probabilities."
        )

    return probability


def compute_region_weight(
    labels,
    calibrated_probability,
):
    auc = float(
        roc_auc_score(
            labels,
            calibrated_probability,
        )
    )

    reliability = max(
        auc - 0.5,
        WEIGHT_EPSILON,
    )

    return auc, reliability


def compute_metrics(
    labels,
    probability,
    threshold=DECISION_THRESHOLD,
):
    labels = np.asarray(
        labels,
        dtype=int,
    )

    probability = np.asarray(
        probability,
        dtype=float,
    )

    prediction = (
        probability
        >= threshold
    ).astype(int)

    tn, fp, fn, tp = (
        confusion_matrix(
            labels,
            prediction,
            labels=[0, 1],
        ).ravel()
    )

    return {
        "n": int(len(labels)),
        "threshold": float(threshold),
        "accuracy": float(
            accuracy_score(labels, prediction)
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                labels,
                prediction,
            )
        ),
        "precision": float(
            precision_score(
                labels,
                prediction,
                zero_division=0,
            )
        ),
        "recall": float(
            recall_score(
                labels,
                prediction,
                zero_division=0,
            )
        ),
        "specificity": float(
            tn / (tn + fp)
            if (tn + fp)
            else np.nan
        ),
        "f1": float(
            f1_score(
                labels,
                prediction,
                zero_division=0,
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                labels,
                probability,
            )
        ),
        "pr_auc": float(
            average_precision_score(
                labels,
                probability,
            )
        ),
        "brier_score": float(
            brier_score_loss(
                labels,
                probability,
            )
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

In [ ]:
# ============================================================
# 8) VISUALIZATION HELPERS
# ============================================================

def plot_weights(
    family,
    validation_auc,
    weights,
    figure_dir,
):
    regions = ["eye", "brow", "mouth"]
    labels = ["Eye", "Brow", "Mouth"]

    fig, ax = plt.subplots(figsize=(9, 6))

    bars = ax.bar(
        labels,
        [weights[r] for r in regions],
    )

    ax.set_title(
        f"{family} — Calibration-Aware Validation Weights",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )

    ax.set_ylabel(
        "Normalized Weight",
        fontsize=11,
    )

    ax.set_ylim(0, 1)
    ax.grid(axis="y", alpha=0.25)

    for bar, region in zip(
        bars,
        regions,
    ):
        ax.text(
            bar.get_x()
            + bar.get_width() / 2,
            bar.get_height(),
            (
                f"w={weights[region]:.3f}\n"
                f"Val AUC={validation_auc[region]:.3f}"
            ),
            ha="center",
            va="bottom",
            fontsize=10,
        )

    fig.tight_layout()

    save_figure(
        fig,
        figure_dir / "validation_weights",
    )


def plot_reliability_before_after(
    validation_df,
    region,
    calibrated_col,
    figure_dir,
    family,
):
    score_col = f"score_{region}"

    fig, ax = plt.subplots(
        figsize=(8, 6)
    )

    ax.scatter(
        validation_df[score_col],
        validation_df[calibrated_col],
        alpha=0.55,
    )

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1.5,
        label="Identity",
    )

    ax.set_title(
        f"{family} — {region.title()} Calibration Mapping",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )

    ax.set_xlabel(
        "Original Model Probability",
        fontsize=11,
    )

    ax.set_ylabel(
        "Calibrated P(FAKE)",
        fontsize=11,
    )

    ax.legend(frameon=True)
    ax.grid(alpha=0.25)

    fig.tight_layout()

    save_figure(
        fig,
        figure_dir
        / f"{region}_calibration_mapping",
    )


def plot_roc_comparison(
    test_df,
    family,
    figure_dir,
):
    y = test_df["label"].to_numpy()

    series = {
        "Eye calibrated": "cal_p_eye",
        "Brow calibrated": "cal_p_brow",
        "Mouth calibrated": "cal_p_mouth",
        "Calibrated weighted fusion": "fusion_probability",
    }

    fig, ax = plt.subplots(
        figsize=(9, 7)
    )

    for name, column in series.items():
        probability = (
            test_df[column]
            .to_numpy()
        )

        fpr, tpr, _ = roc_curve(
            y,
            probability,
        )

        auc = roc_auc_score(
            y,
            probability,
        )

        ax.plot(
            fpr,
            tpr,
            linewidth=2,
            label=f"{name} (AUC={auc:.3f})",
        )

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1.5,
        label="Chance",
    )

    ax.set_title(
        f"{family} — Test ROC Comparison",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )

    ax.set_xlabel(
        "False Positive Rate",
        fontsize=11,
    )

    ax.set_ylabel(
        "True Positive Rate",
        fontsize=11,
    )

    ax.legend(frameon=True)
    ax.grid(alpha=0.25)

    fig.tight_layout()

    save_figure(
        fig,
        figure_dir
        / "test_roc_comparison",
    )


def plot_pr_comparison(
    test_df,
    family,
    figure_dir,
):
    y = test_df["label"].to_numpy()

    series = {
        "Eye calibrated": "cal_p_eye",
        "Brow calibrated": "cal_p_brow",
        "Mouth calibrated": "cal_p_mouth",
        "Calibrated weighted fusion": "fusion_probability",
    }

    fig, ax = plt.subplots(
        figsize=(9, 7)
    )

    for name, column in series.items():
        probability = (
            test_df[column]
            .to_numpy()
        )

        precision, recall, _ = (
            precision_recall_curve(
                y,
                probability,
            )
        )

        ap = average_precision_score(
            y,
            probability,
        )

        ax.plot(
            recall,
            precision,
            linewidth=2,
            label=f"{name} (AP={ap:.3f})",
        )

    ax.set_title(
        f"{family} — Test Precision–Recall Comparison",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )

    ax.set_xlabel("Recall", fontsize=11)
    ax.set_ylabel("Precision", fontsize=11)

    ax.legend(frameon=True)
    ax.grid(alpha=0.25)

    fig.tight_layout()

    save_figure(
        fig,
        figure_dir
        / "test_precision_recall_comparison",
    )


def plot_confusion(
    test_df,
    family,
    figure_dir,
):
    y = test_df["label"].to_numpy()

    pred = (
        test_df["fusion_probability"]
        .to_numpy()
        >= DECISION_THRESHOLD
    ).astype(int)

    matrix = confusion_matrix(
        y,
        pred,
        labels=[0, 1],
    )

    fig, ax = plt.subplots(
        figsize=(7, 6)
    )

    image = ax.imshow(matrix)

    ax.set_xticks(
        [0, 1],
        labels=["REAL", "FAKE"],
    )

    ax.set_yticks(
        [0, 1],
        labels=["REAL", "FAKE"],
    )

    ax.set_xlabel(
        "Predicted Class",
        fontsize=11,
    )

    ax.set_ylabel(
        "True Class",
        fontsize=11,
    )

    ax.set_title(
        f"{family} — Calibrated Weighted Fusion Confusion Matrix",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )

    for row in range(2):
        for column in range(2):
            ax.text(
                column,
                row,
                str(int(matrix[row, column])),
                ha="center",
                va="center",
                fontsize=12,
            )

    fig.colorbar(image, ax=ax)
    fig.tight_layout()

    save_figure(
        fig,
        figure_dir
        / "fusion_confusion_matrix",
    )

In [ ]:
# ============================================================
# 9) RUN CALIBRATED WEIGHTED FUSION
# ============================================================

family_status_rows = []
metric_rows = []
weight_rows = []

for family, cfg in MODEL_FAMILIES.items():
    print("\n" + "=" * 90)
    print("MODEL FAMILY:", family)
    print("=" * 90)

    family_dir = RUN_DIR / family
    figure_dir = family_dir / "figures"

    validation_paths = (
        VALIDATION_PATHS[family]
    )

    missing_regions = [
        region
        for region in [
            "eye",
            "brow",
            "mouth",
        ]
        if validation_paths[region] is None
    ]

    if missing_regions:
        status = {
            "model_family": family,
            "status": "BLOCKED_MISSING_VALIDATION",
            "missing_regions": ",".join(
                missing_regions
            ),
        }

        family_status_rows.append(status)

        atomic_write_json(
            {
                **status,
                "reason": (
                    "Calibration must be fit on validation scores. "
                    "TEST calibration is prohibited."
                ),
            },
            family_dir
            / "audit"
            / "BLOCKING_REPORT.json",
        )

        print(
            "BLOCKED — missing validation predictions:",
            missing_regions,
        )

        continue

    try:
        # ----------------------------------------------------
        # Validation alignment
        # ----------------------------------------------------
        validation_df, validation_audit = (
            align_three(
                validation_paths["eye"],
                validation_paths["brow"],
                validation_paths["mouth"],
            )
        )

        # ----------------------------------------------------
        # Test alignment
        # ----------------------------------------------------
        test_df, test_audit = (
            align_three(
                cfg["eye_test"],
                cfg["brow_test"],
                cfg["mouth_test"],
            )
        )

        # Explicit split leakage check by normalized frame key.
        overlap = (
            set(validation_df["fusion_key"])
            & set(test_df["fusion_key"])
        )

        if overlap:
            raise RuntimeError(
                f"{family}: validation/test overlap detected: "
                f"{len(overlap)} frames."
            )

        calibrators = {}
        validation_auc = {}
        reliability = {}
        weights = {}

        # ----------------------------------------------------
        # Region calibration, validation only
        # ----------------------------------------------------
        for region in [
            "eye",
            "brow",
            "mouth",
        ]:
            score_column = (
                f"score_{region}"
            )

            calibrator = (
                fit_platt_calibrator(
                    validation_df[
                        score_column
                    ],
                    validation_df[
                        "label"
                    ],
                )
            )

            calibrators[region] = (
                calibrator
            )

            validation_df[
                f"cal_p_{region}"
            ] = calibrate(
                calibrator,
                validation_df[
                    score_column
                ],
            )

            test_df[
                f"cal_p_{region}"
            ] = calibrate(
                calibrator,
                test_df[
                    score_column
                ],
            )

            auc, rel = (
                compute_region_weight(
                    validation_df[
                        "label"
                    ],
                    validation_df[
                        f"cal_p_{region}"
                    ],
                )
            )

            validation_auc[region] = auc
            reliability[region] = rel

        denominator = float(
            sum(
                reliability.values()
            )
        )

        weights = {
            region: float(
                reliability[region]
                / denominator
            )
            for region in reliability
        }

        if not math.isclose(
            sum(weights.values()),
            1.0,
            rel_tol=1e-9,
            abs_tol=1e-9,
        ):
            raise RuntimeError(
                f"{family}: weights do not sum to one."
            )

        # ----------------------------------------------------
        # Final calibrated weighted score
        # ----------------------------------------------------
        test_df[
            "fusion_probability"
        ] = (
            weights["eye"]
            * test_df["cal_p_eye"]
            + weights["brow"]
            * test_df["cal_p_brow"]
            + weights["mouth"]
            * test_df["cal_p_mouth"]
        )

        validation_df[
            "fusion_probability"
        ] = (
            weights["eye"]
            * validation_df["cal_p_eye"]
            + weights["brow"]
            * validation_df["cal_p_brow"]
            + weights["mouth"]
            * validation_df["cal_p_mouth"]
        )

        # ----------------------------------------------------
        # Test metrics
        # ----------------------------------------------------
        evaluation_columns = {
            "eye_calibrated": "cal_p_eye",
            "brow_calibrated": "cal_p_brow",
            "mouth_calibrated": "cal_p_mouth",
            "fusion": "fusion_probability",
        }

        family_metrics = []

        for evaluation, column in (
            evaluation_columns.items()
        ):
            metrics = compute_metrics(
                test_df["label"],
                test_df[column],
                threshold=DECISION_THRESHOLD,
            )

            row = {
                "model_family": family,
                "method": METHOD_NAME,
                "evaluation": evaluation,
                "threshold_source": (
                    "fixed_predefined_0.50"
                ),
                "calibration_source": (
                    "validation_only"
                ),
                "weight_source": (
                    "validation_roc_auc_only"
                ),
                "weight_eye": weights["eye"],
                "weight_brow": weights["brow"],
                "weight_mouth": weights["mouth"],
                **metrics,
            }

            family_metrics.append(row)
            metric_rows.append(row)

        family_metrics = pd.DataFrame(
            family_metrics
        )

        # ----------------------------------------------------
        # Save
        # ----------------------------------------------------
        atomic_write_csv(
            validation_df,
            family_dir
            / "predictions"
            / "aligned_validation_calibrated.csv",
        )

        atomic_write_csv(
            test_df,
            family_dir
            / "predictions"
            / "aligned_test_calibrated_fusion.csv",
        )

        atomic_write_csv(
            family_metrics,
            family_dir
            / "metrics"
            / "test_metrics.csv",
        )

        for region in [
            "eye",
            "brow",
            "mouth",
        ]:
            model = calibrators[region]

            weight_rows.append({
                "model_family": family,
                "region": region,
                "validation_roc_auc": (
                    validation_auc[region]
                ),
                "reliability": (
                    reliability[region]
                ),
                "normalized_weight": (
                    weights[region]
                ),
                "calibrator_coef": float(
                    model.coef_[0, 0]
                ),
                "calibrator_intercept": float(
                    model.intercept_[0]
                ),
            })

        atomic_write_json(
            {
                "run_id": RUN_ID,
                "model_family": family,
                "method": METHOD_NAME,
                "decision_threshold": (
                    DECISION_THRESHOLD
                ),
                "calibration": (
                    "1D Logistic/Platt on validation only"
                ),
                "validation_auc": (
                    validation_auc
                ),
                "reliability": (
                    reliability
                ),
                "weights": weights,
                "validation_paths": {
                    region: str(
                        validation_paths[
                            region
                        ]
                    )
                    for region in [
                        "eye",
                        "brow",
                        "mouth",
                    ]
                },
                "test_paths": {
                    region: str(
                        cfg[
                            f"{region}_test"
                        ]
                    )
                    for region in [
                        "eye",
                        "brow",
                        "mouth",
                    ]
                },
                "validation_audit": (
                    validation_audit
                ),
                "test_audit": (
                    test_audit
                ),
                "test_used_for_calibration": False,
                "test_used_for_weight_selection": False,
                "test_used_for_threshold_selection": False,
            },
            family_dir
            / "audit"
            / "run_audit.json",
        )

        # ----------------------------------------------------
        # Figures
        # ----------------------------------------------------
        figure_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        plot_weights(
            family,
            validation_auc,
            weights,
            figure_dir,
        )

        for region in [
            "eye",
            "brow",
            "mouth",
        ]:
            plot_reliability_before_after(
                validation_df,
                region,
                f"cal_p_{region}",
                figure_dir,
                family,
            )

        plot_roc_comparison(
            test_df,
            family,
            figure_dir,
        )

        plot_pr_comparison(
            test_df,
            family,
            figure_dir,
        )

        plot_confusion(
            test_df,
            family,
            figure_dir,
        )

        fusion_metrics = (
            family_metrics.loc[
                family_metrics[
                    "evaluation"
                ]
                == "fusion"
            ]
            .iloc[0]
        )

        family_status_rows.append({
            "model_family": family,
            "status": "SUCCESS",
            "missing_regions": "",
            "common_validation_frames": int(
                len(validation_df)
            ),
            "common_test_frames": int(
                len(test_df)
            ),
            "fusion_roc_auc": float(
                fusion_metrics[
                    "roc_auc"
                ]
            ),
            "fusion_pr_auc": float(
                fusion_metrics[
                    "pr_auc"
                ]
            ),
            "fusion_f1": float(
                fusion_metrics[
                    "f1"
                ]
            ),
        })

        print(
            "SUCCESS | "
            f"VAL={len(validation_df)} | "
            f"TEST={len(test_df)} | "
            f"ROC-AUC={fusion_metrics['roc_auc']:.4f} | "
            f"F1={fusion_metrics['f1']:.4f}"
        )

    except Exception as exc:
        family_status_rows.append({
            "model_family": family,
            "status": "FAILED",
            "missing_regions": "",
            "error": (
                f"{type(exc).__name__}: {exc}"
            ),
        })

        atomic_write_json(
            {
                "model_family": family,
                "status": "FAILED",
                "error_type": (
                    type(exc).__name__
                ),
                "error": str(exc),
            },
            family_dir
            / "audit"
            / "FAILED.json",
        )

        print(
            "FAILED:",
            type(exc).__name__,
            exc,
        )


status_df = pd.DataFrame(
    family_status_rows
)

metrics_df = pd.DataFrame(
    metric_rows
)

weights_df = pd.DataFrame(
    weight_rows
)

atomic_write_csv(
    status_df,
    RUN_DIR
    / "family_status.csv",
)

if not metrics_df.empty:
    atomic_write_csv(
        metrics_df,
        RUN_DIR
        / "metrics"
        / "all_model_families_metrics.csv",
    )

if not weights_df.empty:
    atomic_write_csv(
        weights_df,
        RUN_DIR
        / "metrics"
        / "calibration_and_weights.csv",
    )

display(status_df)

In [ ]:
# ============================================================
# 10) CROSS-FAMILY COMPARISON FOR SUCCESSFUL FAMILIES
# ============================================================

if not metrics_df.empty:
    fusion_only = (
        metrics_df.loc[
            metrics_df["evaluation"]
            == "fusion"
        ]
        .copy()
    )

    if not fusion_only.empty:
        plot_data = (
            fusion_only[
                [
                    "model_family",
                    "roc_auc",
                    "pr_auc",
                    "balanced_accuracy",
                    "f1",
                ]
            ]
            .set_index(
                "model_family"
            )
        )

        fig, ax = plt.subplots(
            figsize=(12, 7)
        )

        plot_data.plot(
            kind="bar",
            ax=ax,
        )

        ax.set_title(
            "Calibrated Weighted Fusion — Model Family Comparison",
            fontsize=14,
            fontweight="bold",
            pad=12,
        )

        ax.set_xlabel(
            "Model Family",
            fontsize=11,
        )

        ax.set_ylabel(
            "Score",
            fontsize=11,
        )

        ax.set_ylim(
            0,
            1.05,
        )

        ax.tick_params(
            axis="x",
            rotation=15,
        )

        ax.legend(
            [
                "ROC-AUC",
                "PR-AUC",
                "Balanced Accuracy",
                "F1",
            ],
            frameon=True,
        )

        ax.grid(
            axis="y",
            alpha=0.25,
        )

        fig.tight_layout()

        save_figure(
            fig,
            RUN_DIR
            / "figures"
            / "model_family_comparison",
        )

        display(
            fusion_only
        )
else:
    print(
        "No family completed calibrated fusion yet."
    )

In [ ]:
# ============================================================
# 11) FINAL AUDIT + OUTPUT MANIFEST
# ============================================================

manifest_rows = []

for path in sorted(
    RUN_DIR.rglob("*")
):
    if not path.is_file():
        continue

    row = {
        "relative_path": str(
            path.relative_to(
                RUN_DIR
            )
        ),
        "size_bytes": int(
            path.stat().st_size
        ),
        "suffix": (
            path.suffix
            .lower()
        ),
    }

    if (
        path.suffix.lower()
        == ".png"
    ):
        with Image.open(
            path
        ) as image:
            row["width_px"] = int(
                image.width
            )

            row["height_px"] = int(
                image.height
            )

            if min(
                image.size
            ) < MIN_FIGURE_SHORT_EDGE_PX:
                raise RuntimeError(
                    f"Low-resolution figure: "
                    f"{path} -> {image.size}"
                )

    manifest_rows.append(
        row
    )


manifest = pd.DataFrame(
    manifest_rows
)

atomic_write_csv(
    manifest,
    RUN_DIR
    / "output_manifest.csv",
)


success_count = int(
    (
        status_df[
            "status"
        ]
        == "SUCCESS"
    )
    .sum()
)

blocked_count = int(
    (
        status_df[
            "status"
        ]
        == "BLOCKED_MISSING_VALIDATION"
    )
    .sum()
)

failed_count = int(
    (
        status_df[
            "status"
        ]
        == "FAILED"
    )
    .sum()
)


final_summary = {
    "run_id": RUN_ID,
    "method": METHOD_NAME,
    "success_count": success_count,
    "blocked_missing_validation_count": (
        blocked_count
    ),
    "failed_count": failed_count,
    "decision_threshold": (
        DECISION_THRESHOLD
    ),
    "calibration_source": (
        "validation_only"
    ),
    "weight_source": (
        "validation_roc_auc_only"
    ),
    "test_used_for_calibration": False,
    "test_used_for_weight_selection": False,
    "test_used_for_threshold_selection": False,
}

atomic_write_json(
    final_summary,
    RUN_DIR
    / "run_summary.json",
)


print("=" * 90)
print("FINAL RUN SUMMARY")
print("=" * 90)

print(
    f"SUCCESS : {success_count}/3"
)

print(
    f"BLOCKED : {blocked_count}/3"
)

print(
    f"FAILED  : {failed_count}/3"
)

print(
    f"OUTPUT  : {RUN_DIR}"
)

display(
    status_df
)

## Beklenen davranış

Şu anda validation prediction generator'da **SwinV2 Tiny / Eye-Brow-Mouth**
başarıyla üretildiği için bu aile doğrudan calibrated weighted fusion yapabilir.

EfficientNet-B0 ve SwinV2+Texture için üç bölgenin validation prediction dosyaları
henüz yoksa notebook bu aileleri **BLOCKED_MISSING_VALIDATION** olarak işaretler.
Bu bilinçli bir quality gate'tir; TEST üzerinde calibration fit edilmez.

Bu nedenle notebook iki şeyi aynı anda sağlar:

- hazır ailelerde üçüncü fusion yönteminin gerçek sonucunu üretir;
- eksik ailelerde akademik olarak hatalı bir shortcut kullanmak yerine eksik
  validation girdisini açıkça gösterir.